### Hyperparameter tuning: test 1 (New tts file)

In [7]:
import h5py
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report, cohen_kappa_score)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils import resample
import time

# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0,0]
    fp = cm[0,1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- Binary SVM classifier --------------------
def svm_classifier(X_train, y_train, X_test, y_test, use_scaling=True, max_iter=10000, C=1.0, tol=1e-4):
    print("\n--- SVM for Sleep vs Awake classification ---")
    
    # Scale features
    if use_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        X_train_scaled, X_test_scaled = X_train, X_test
        print("Data scaling skipped.")
    
    # Initialize classifier
    clf = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced', random_state=42, tol=tol)
    
    # Fit
    start_time = time.time()
    clf.fit(X_train_scaled, y_train)
    print(f"SVM fitted in {time.time() - start_time:.2f} seconds.")
    
    # Predict
    y_pred = clf.predict(X_test_scaled)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_test, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)
    specificity = specificity_score(y_test, y_pred)
    cohen_kappa = cohen_kappa_score(y_test, y_pred)
    
    print("\nEvaluation Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"Cohen Kappa: {cohen_kappa:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Sleep','Wake'], zero_division=0))
    
    return clf

# -------------------- RandomizedSearchCV for hyperparameter tuning --------------------
def tune_svm(X_train, y_train):
    print("\n🔍 Running Random Search for Hyperparameter Tuning (Binary)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    svm = LinearSVC(class_weight="balanced", max_iter=20000, random_state=42)
    
    param_dist = {
        "C": np.logspace(-5, 5, 20),      #Wider range
        "tol": np.logspace(-6, -3, 30)    #finer granuilarity
    }
    
    search = RandomizedSearchCV(
        svm,
        param_distributions=param_dist,
        n_iter=20,
        cv=3,
        scoring="f1_macro",
        verbose=1,
        n_jobs=-1,
        random_state=42
    )
    
    search.fit(X_train_scaled, y_train)
    
    print(f"\n✅ Best hyperparameters: {search.best_params_}")
    print(f"🏅 Best CV score (F1 macro): {search.best_score_:.4f}")
    
    return search.best_params_

# -------------------- Load HDF5 train/test split --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# --- New 80/20 split from combined data ---
from sklearn.model_selection import train_test_split

X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
# -------------------- Filter unknown labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
# Original Wake = 0, Sleep stages = [1,2,3,5]
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)

sleep_stages = [1,2,3,5]
y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0

# Map original Wake (0) → 1
y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1

print("Unique train labels:", np.unique(y_train_bin, return_counts=True))
print("Unique test labels:", np.unique(y_test_bin, return_counts=True))

# -------------------- Upsample Wake class if needed --------------------
X_train_sleep = X_train[y_train_bin == 0]
y_train_sleep = y_train_bin[y_train_bin == 0]
X_train_wake = X_train[y_train_bin == 1]
y_train_wake = y_train_bin[y_train_bin == 1]

if len(X_train_sleep) > len(X_train_wake):
    X_wake_upsampled, y_wake_upsampled = resample(
        X_train_wake, y_train_wake,
        replace=True,
        n_samples=len(X_train_sleep),
        random_state=42
    )
    X_train_balanced = np.vstack([X_train_sleep, X_wake_upsampled])
    y_train_balanced = np.hstack([y_train_sleep, y_wake_upsampled])
else:
    X_train_balanced = np.vstack([X_train_sleep, X_train_wake])
    y_train_balanced = np.hstack([y_train_sleep, y_train_wake])

print(f"Balanced training set: {X_train_balanced.shape[0]} samples")

# -------------------- Run Random Search --------------------
best_params_binary = tune_svm(X_train_balanced, y_train_balanced)

# -------------------- Train final SVM --------------------
svm_binary = svm_classifier(
    X_train_balanced, y_train_balanced,
    X_test, y_test_bin,
    use_scaling=True,
    max_iter=20000,
    C=best_params_binary['C'],
    tol=best_params_binary['tol']
)

Training samples: 46398, Test samples: 11600
Unique train labels: (array([0, 1]), array([38831,  7485]))
Unique test labels: (array([0, 1]), array([9708, 1872]))
Balanced training set: 77662 samples

🔍 Running Random Search for Hyperparameter Tuning (Binary)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✅ Best hyperparameters: {'tol': np.float64(0.00038566204211634724), 'C': np.float64(29763.51441631313)}
🏅 Best CV score (F1 macro): 0.8715

--- SVM for Sleep vs Awake classification ---
Data scaled with StandardScaler.
SVM fitted in 1.71 seconds.

Evaluation Metrics:
Accuracy: 0.8948
Precision: 0.6345
Recall: 0.8243
F1 Score: 0.7170
Specificity: 0.9084
Cohen Kappa: 0.6538

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.96      0.91      0.94      9708
        Wake       0.63      0.82      0.72      1872

    accuracy                           0.89     11580
   macro avg       0.80      0.87      0.83     11580
weigh

### Hypertuning: Test 2 (Adasyn)

In [1]:
import h5py
import numpy as np
import time
from collections import Counter

from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split

from imblearn.over_sampling import ADASYN

# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- Binary SVM classifier --------------------
def svm_classifier(X_train, y_train, X_test, y_test,
                   use_scaling=True, max_iter=10000, C=1, tol=4e-4,
                   class_weight='balanced'):
    print("\n--- SVM for Sleep vs Wake classification ---")

    # Scale features
    if use_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        X_train_scaled, X_test_scaled = X_train, X_test
        print("Data scaling skipped.")

    # Initialize SVM with class imbalance
    clf = LinearSVC(
        C=C,
        max_iter=max_iter,
        class_weight=class_weight,
        random_state=42,
        tol=tol
    )

    # Fit
    start_time = time.time()
    clf.fit(X_train_scaled, y_train)
    print(f"SVM fitted in {time.time() - start_time:.2f} seconds.")

    # Predict
    y_pred = clf.predict(X_test_scaled)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_test, y_pred, average='binary', zero_division=0)  # Wake sensitivity
    f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)
    sensitivity_sleep = recall_score(y_test, y_pred, pos_label=0)  # Sleep sensitivity
    specificity = specificity_score(y_test, y_pred)
    cohen_kappa = cohen_kappa_score(y_test, y_pred)

    print("\nEvaluation Metrics:")
    print(f"Accuracy:            {accuracy:.4f}")
    print(f"Precision (Wake):    {precision:.4f}")
    print(f"Recall (Wake):       {recall:.4f}")
    print(f"Sensitivity (Sleep): {sensitivity_sleep:.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:            {f1:.4f}")
    print(f"Cohen’s Kappa:       {cohen_kappa:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Sleep', 'Wake'], zero_division=0))

    return clf

# -------------------- Load HDF5 train/test split --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# -------------------- New 80/20 split --------------------
X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# -------------------- Filter unknown labels --------------------
valid_labels = [0, 1, 2, 3, 5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
sleep_stages = [1, 2, 3, 5]
y_train_bin = np.where(np.isin(y_train, sleep_stages), 0, 1)
y_test_bin  = np.where(np.isin(y_test, sleep_stages), 0, 1)

print("Train distribution before ADASYN:", np.unique(y_train_bin, return_counts=True))

# -------------------- Apply ADASYN --------------------
adasyn = ADASYN(random_state=42)
X_train_res, y_train_res = adasyn.fit_resample(X_train, y_train_bin)

print("Train distribution after ADASYN:", Counter(y_train_res))

# -------------------- Train SVM --------------------
svm_model = svm_classifier(
    X_train_res, y_train_res,
    X_test, y_test_bin,
    use_scaling=True,
    max_iter=10000,
    class_weight='balanced'
)


Training samples: 46398, Test samples: 11600
Train distribution before ADASYN: (array([0, 1]), array([38831,  7485]))
Train distribution after ADASYN: Counter({np.int64(1): 39264, np.int64(0): 38831})

--- SVM for Sleep vs Wake classification ---
Data scaled with StandardScaler.
SVM fitted in 3.11 seconds.

Evaluation Metrics:
Accuracy:            0.8764
Precision (Wake):    0.5782
Recall (Wake):       0.8713
Sensitivity (Sleep): 0.8774
Specificity (Wake):  0.8774
F1 Score:            0.6951
Cohen’s Kappa:       0.6215

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.97      0.88      0.92      9708
        Wake       0.58      0.87      0.70      1872

    accuracy                           0.88     11580
   macro avg       0.78      0.87      0.81     11580
weighted avg       0.91      0.88      0.89     11580



# Class imbalance (Test 2) BEST

In [6]:
import h5py
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score
)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import ADASYN

# -------------------- Metrics --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)  # Wake sensitivity
    f1 = f1_score(y_true, y_pred, zero_division=0)
    sensitivity_sleep = recall_score(y_true, y_pred, pos_label=0)
    specificity = specificity_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    
    print("\nEvaluation Metrics:")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Precision (Wake):    {precision:.4f}")
    print(f"Recall (Wake):       {recall:.4f}")
    print(f"Sensitivity (Sleep): {sensitivity_sleep:.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:            {f1:.4f}")
    print(f"Cohen’s Kappa:       {kappa:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Sleep','Wake'], zero_division=0))


# -------------------- Load data --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test  = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test  = f['y_test'][:]

# Combine and split 80/20
X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# Filter unknown labels
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# Binary mapping: Sleep=0, Wake=1
sleep_stages = [1,2,3,5]
y_train_bin = np.where(np.isin(y_train, sleep_stages), 0, 1)
y_test_bin  = np.where(np.isin(y_test, sleep_stages), 0, 1)

print("Train distribution:", np.unique(y_train_bin, return_counts=True))
print("Test distribution :", np.unique(y_test_bin, return_counts=True))

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


# -------------------- Class imbalance options --------------------
# Option 1: ADASYN oversampling
adasyn = ADASYN(random_state=42)
X_train_bal, y_train_bal = adasyn.fit_resample(X_train_scaled, y_train_bin)
print("After ADASYN:", Counter(y_train_bal))

# Option 2: Use class_weight='balanced' (no resampling needed)
#X_train_bal, y_train_bal = X_train_scaled, y_train_bin


# -------------------- Choose classifier --------------------
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Example 1: SVM
clf = LinearSVC(max_iter=10000, class_weight='balanced', random_state=42)

# Example 2: Random Forest
#clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)

# Example 3: Logistic Regression
# clf = LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)

# Fit
clf.fit(X_train_bal, y_train_bal)

# Predict
y_pred = clf.predict(X_test_scaled)

# Evaluate
evaluate_model(y_test_bin, y_pred)


Train distribution: (array([0, 1]), array([38831,  7485]))
Test distribution : (array([0, 1]), array([9708, 1872]))
After ADASYN: Counter({np.int64(1): 39162, np.int64(0): 38831})

Evaluation Metrics:
Accuracy:            0.8782
Precision (Wake):    0.5832
Recall (Wake):       0.8654
Sensitivity (Sleep): 0.8807
Specificity (Wake):  0.8807
F1 Score:            0.6968
Cohen’s Kappa:       0.6242

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.97      0.88      0.92      9708
        Wake       0.58      0.87      0.70      1872

    accuracy                           0.88     11580
   macro avg       0.78      0.87      0.81     11580
weighted avg       0.91      0.88      0.89     11580



### Class Imbalance: Class weights


In [5]:
import h5py
import numpy as np
import time
from collections import Counter

from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score
)
from sklearn.model_selection import train_test_split


# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0


# -------------------- Binary SVM classifier --------------------
def svm_classifier(
    X_train, y_train,
    X_test, y_test,
    use_scaling=True,
    max_iter=10000,
    C=1.0,
    tol=1e-4,
    class_weight=None
):
    print("\n--- SVM for Sleep vs Wake classification ---")

    print("Using manual class weights:", class_weight)

    # ---- Scaling ----
    if use_scaling:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        print("Data scaling skipped.")

    # ---- SVM ----
    clf = LinearSVC(
        C=C,
        max_iter=max_iter,
        tol=tol,
        class_weight=class_weight,
        random_state=42
    )

    start_time = time.time()
    clf.fit(X_train, y_train)
    print(f"SVM fitted in {time.time() - start_time:.2f} seconds.")

    # ---- Prediction ----
    y_pred = clf.predict(X_test)

    # ---- Metrics ----
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)   # Wake recall
    f1 = f1_score(y_test, y_pred, zero_division=0)
    sensitivity_sleep = recall_score(y_test, y_pred, pos_label=0)
    specificity = specificity_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)

    print("\nEvaluation Metrics:")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Precision (Wake):    {precision:.4f}")
    print(f"Recall (Wake):       {recall:.4f}")
    print(f"Sensitivity (Sleep): {sensitivity_sleep:.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:            {f1:.4f}")
    print(f"Cohen’s Kappa:       {kappa:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred,
        target_names=["Sleep", "Wake"],
        zero_division=0
    ))

    return clf


# -------------------- Load HDF5 --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, "r") as f:
    X_train_old = f["X_train"][:]
    X_test_old  = f["X_test"][:]
    y_train_old = f["y_train"][:]
    y_test_old  = f["y_test"][:]

# -------------------- Re-split 80/20 --------------------
X_all = np.concatenate([X_train_old, X_test_old])
y_all = np.concatenate([y_train_old, y_test_old])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=42
)

print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")


# -------------------- Filter valid sleep labels --------------------
valid_labels = [0, 1, 2, 3, 5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
y_train = y_train[train_mask]
X_test  = X_test[test_mask]
y_test  = y_test[test_mask]


# -------------------- Binary mapping: Sleep=0, Wake=1 --------------------
sleep_stages = [1, 2, 3, 5]

y_train_bin = np.where(np.isin(y_train, sleep_stages), 0, 1)
y_test_bin  = np.where(np.isin(y_test, sleep_stages), 0, 1)

print("Train distribution:", np.unique(y_train_bin, return_counts=True))
print("Test distribution :", np.unique(y_test_bin, return_counts=True))


# -------------------- MANUAL CLASS WEIGHTS --------------------
# Wake is rare → penalize Wake errors more
class_weight = {0: 1, 1: 7}
print("\nUsing manual class weights:", class_weight)


# -------------------- Train SVM --------------------
svm_model = svm_classifier(
    X_train, y_train_bin,
    X_test, y_test_bin,
    use_scaling=True,
    max_iter=10000,
    C=1.0,
    tol=1e-4,
    class_weight=class_weight
)


Train samples: 46398, Test samples: 11600
Train distribution: (array([0, 1]), array([38831,  7485]))
Test distribution : (array([0, 1]), array([9708, 1872]))

Using manual class weights: {0: 1, 1: 7}

--- SVM for Sleep vs Wake classification ---
Using manual class weights: {0: 1, 1: 7}
Data scaled with StandardScaler.
SVM fitted in 2.22 seconds.

Evaluation Metrics:
Accuracy:            0.8813
Precision (Wake):    0.5916
Recall (Wake):       0.8590
Sensitivity (Sleep): 0.8857
Specificity (Wake):  0.8857
F1 Score:            0.7007
Cohen’s Kappa:       0.6298

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.97      0.89      0.93      9708
        Wake       0.59      0.86      0.70      1872

    accuracy                           0.88     11580
   macro avg       0.78      0.87      0.81     11580
weighted avg       0.91      0.88      0.89     11580



### SMOTE

In [6]:
import h5py
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report, cohen_kappa_score)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from imblearn.over_sampling import SMOTE
import time
from collections import Counter


# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0,0]
    fp = cm[0,1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- Binary SVM classifier --------------------
def svm_classifier(X_train, y_train, X_test, y_test, use_scaling=True, max_iter=10000, C=1.0, tol=1e-4):
    print("\n--- SVM for Sleep vs Wake classification ---")
    
    # Scale features
    if use_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        X_train_scaled, X_test_scaled = X_train, X_test
        print("Data scaling skipped.")
    
    # Initialize classifier
    clf = LinearSVC(C=C, max_iter=max_iter, class_weight=None, random_state=42, tol=tol)
    
    # Fit
    start_time = time.time()
    clf.fit(X_train_scaled, y_train)
    print(f"SVM fitted in {time.time() - start_time:.2f} seconds.")
    
    # Predict
    y_pred = clf.predict(X_test_scaled)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_test, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)
    specificity = specificity_score(y_test, y_pred)
    cohen_kappa = cohen_kappa_score(y_test, y_pred)
    
    print("\nEvaluation Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"Cohen Kappa: {cohen_kappa:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Sleep','Wake'], zero_division=0))
    
    return clf

# -------------------- RandomizedSearchCV for hyperparameter tuning --------------------
def tune_svm(X_train, y_train):
    print("\n🔍 Running Random Search for Hyperparameter Tuning (Binary)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    svm = LinearSVC(class_weight=None, max_iter=20000, random_state=42)
    
    param_dist = {
        "C": np.logspace(-5, 5, 20),
        "tol": np.logspace(-6, -3, 30)
    }
    
    search = RandomizedSearchCV(
        svm,
        param_distributions=param_dist,
        n_iter=20,
        cv=3,
        scoring="f1_macro",
        verbose=1,
        n_jobs=-1,
        random_state=42
    )
    
    search.fit(X_train_scaled, y_train)
    
    print(f"\n✅ Best hyperparameters: {search.best_params_}")
    print(f"🏅 Best CV score (F1 macro): {search.best_score_:.4f}")
    
    return search.best_params_

# -------------------- Load HDF5 train/test split --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# --- New 80/20 split from combined data ---
X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# -------------------- Filter unknown labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
sleep_stages = [1,2,3,5]
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)

y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0

y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1

print("Unique train labels:", np.unique(y_train_bin, return_counts=True))
print("Unique test labels:", np.unique(y_test_bin, return_counts=True))

# -------------------- Apply SMOTE to training data --------------------
print("\nApplying SMOTE to training data...")
sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train_bin)
print("Resampled training dataset shape %s" % Counter(y_train_smote))

# -------------------- Run Random Search --------------------
best_params_binary = tune_svm(X_train_smote, y_train_smote)

# -------------------- Train final SVM --------------------
svm_binary = svm_classifier(
    X_train_smote, y_train_smote,
    X_test, y_test_bin,
    use_scaling=True,
    max_iter=20000,
    C=best_params_binary['C'],
    tol=best_params_binary['tol']
)


Training samples: 46398, Test samples: 11600
Unique train labels: (array([0, 1]), array([38831,  7485]))
Unique test labels: (array([0, 1]), array([9708, 1872]))

Applying SMOTE to training data...
Resampled training dataset shape Counter({np.int64(0): 38831, np.int64(1): 38831})

🔍 Running Random Search for Hyperparameter Tuning (Binary)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✅ Best hyperparameters: {'tol': np.float64(0.001), 'C': np.float64(69.51927961775606)}
🏅 Best CV score (F1 macro): 0.8788

--- SVM for Sleep vs Wake classification ---
Data scaled with StandardScaler.
SVM fitted in 1.99 seconds.

Evaluation Metrics:
Accuracy: 0.8949
Precision: 0.6339
Recall: 0.8280
F1 Score: 0.7181
Specificity: 0.9078
Cohen Kappa: 0.6549

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.96      0.91      0.94      9708
        Wake       0.63      0.83      0.72      1872

    accuracy                           0.89     1

### Class imbalance: Random oversampler

In [2]:
import h5py
import numpy as np
import time
from collections import Counter

from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score
)
from sklearn.model_selection import train_test_split

from imblearn.over_sampling import RandomOverSampler  # <-- import RandomOverSampler

# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- Binary SVM classifier --------------------
def svm_classifier(
    X_train, y_train,
    X_test, y_test,
    use_scaling=True,
    max_iter=10000,
    C=1.0,
    tol=1e-4
):
    print("\n--- SVM for Sleep vs Wake classification ---")

    # ---- Scaling ----
    if use_scaling:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        print("Data scaling skipped.")

    # ---- SVM ----
    clf = LinearSVC(
        C=C,
        max_iter=max_iter,
        tol=tol,
        random_state=42
    )

    start_time = time.time()
    clf.fit(X_train, y_train)
    print(f"SVM fitted in {time.time() - start_time:.2f} seconds.")

    # ---- Prediction ----
    y_pred = clf.predict(X_test)

    # ---- Metrics ----
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)   # Wake recall
    f1 = f1_score(y_test, y_pred, zero_division=0)
    sensitivity_sleep = recall_score(y_test, y_pred, pos_label=0)
    specificity = specificity_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)

    print("\nEvaluation Metrics:")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Precision (Wake):    {precision:.4f}")
    print(f"Recall (Wake):       {recall:.4f}")
    print(f"Sensitivity (Sleep): {sensitivity_sleep:.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:            {f1:.4f}")
    print(f"Cohen’s Kappa:       {kappa:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred,
        target_names=["Sleep", "Wake"],
        zero_division=0
    ))

    return clf

# -------------------- Load HDF5 --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, "r") as f:
    X_train_old = f["X_train"][:]
    X_test_old  = f["X_test"][:]
    y_train_old = f["y_train"][:]
    y_test_old  = f["y_test"][:]

# -------------------- Re-split 80/20 --------------------
X_all = np.concatenate([X_train_old, X_test_old])
y_all = np.concatenate([y_train_old, y_test_old])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=42
)

print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# -------------------- Filter valid sleep labels --------------------
valid_labels = [0, 1, 2, 3, 5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
y_train = y_train[train_mask]
X_test  = X_test[test_mask]
y_test  = y_test[test_mask]

# -------------------- Binary mapping: Sleep=0, Wake=1 --------------------
sleep_stages = [1, 2, 3, 5]

y_train_bin = np.where(np.isin(y_train, sleep_stages), 0, 1)
y_test_bin  = np.where(np.isin(y_test, sleep_stages), 0, 1)

print("Train distribution before ROS:", np.unique(y_train_bin, return_counts=True))
print("Test distribution :", np.unique(y_test_bin, return_counts=True))

# -------------------- Random Over-Sampling --------------------
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train_bin)
print("Train distribution after ROS:", Counter(y_train_res))

# -------------------- Train SVM --------------------
svm_model = svm_classifier(
    X_train_res, y_train_res,
    X_test, y_test_bin,
    use_scaling=True,
    max_iter=10000,
    C= 1.0,
    tol=1e-4
)


Train samples: 46398, Test samples: 11600
Train distribution before ROS: (array([0, 1]), array([38831,  7485]))
Test distribution : (array([0, 1]), array([9708, 1872]))
Train distribution after ROS: Counter({np.int64(0): 38831, np.int64(1): 38831})

--- SVM for Sleep vs Wake classification ---
Data scaled with StandardScaler.
SVM fitted in 3.50 seconds.

Evaluation Metrics:
Accuracy:            0.8951
Precision (Wake):    0.6352
Recall (Wake):       0.8243
Sensitivity (Sleep): 0.9087
Specificity (Wake):  0.9087
F1 Score:            0.7175
Cohen’s Kappa:       0.6544

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.96      0.91      0.94      9708
        Wake       0.64      0.82      0.72      1872

    accuracy                           0.90     11580
   macro avg       0.80      0.87      0.83     11580
weighted avg       0.91      0.90      0.90     11580

